In [1]:
from portfolio.liabilities import Liabilities
from portfolio.portfolio import *
from assumptions.cma import *
from math import sqrt
from pprint import pprint

## Representing Liabilities

The `Liabilities` object represents pension liabilities based on our capital markets assumptions. This Projects one path of liability growth over any horizon.

In [2]:
l = Liabilities(
    retired_members=RETIRED_MEMBERS,
    active_members=ACTIVE_MEMBERS,
    average_salary=AVG_SALARY,
    min_active_members=MIN_ACTIVE_MEMBERS,
    active_members_decline=ACTIVE_MEMBER_DECLINE,
    retired_members_growth=RETIRED_MEMBERS_GROWTH,
    wage_growth_rate=WAGE_GROWTH,
    starting_duration=INITIAL_DURATION,
    actuarial_df=ACTUARIAL_DF,
    service_cost=SERVICE_COST_RATE,
    starting_liabilities=LIABILITIES,
    starting_benefit=STARTING_BENEFIT,
    benefit_growth_rate=BENEFIT_GROWTH_RATE,
    liabilities_cache={}
)


In [3]:
l.get_closing_liabilities(30)

9407397613.929964

In [4]:
l.benefits_paid(1)

83200000.0

In [5]:
l.get_interest_cost(1)

63600000.0

In [6]:
l.get_service_cost(1)

129364725.0

## Asset Configurations
We have preset Asset configurations for the 9 core classes we currently hold in the portfolio (can easily be expanded with new asset classes). Each contains an expected return, variance, and liquidity rating.

In [7]:
cash = Cash()
can_equity = CanEquity()
us_equity = USEquity()
em_equity = EMEquity()
id_equity = IDEquity()
fixed_income = FixedIncome()
private_equity = PrivateEquity()
infrastructure = Infrastructure()
real_estate = RealEstate()

## Asset Allocation 
The `AssetAlloc` object allows us to build multiple portfolios for testing. You simply create your assets (as above) and assign weights to them.

In [8]:
a = AssetAlloc(
    [
        (cash, 0.0),
        (can_equity, 0.0),
        (us_equity, 0.0),
        (em_equity, 0.0),
        (id_equity, 0.0),
        (fixed_income, 0),
        (private_equity, 1.0),
        (infrastructure, 0.0),
        (real_estate, 0.0),

    ]
)

In [9]:
a.get_expected_return()

0.084

In [10]:
a.get_std_dev()

0.19800051303771027

In [11]:
a.get_sharpe()

0.27272656606559104

## Portfolio
The `Portfolio` object combines the `Liabilities` projection with the `AssetAlloc`. These two components make up the full pension portfolio. This runs background simulations to give us estimates of our portfolio's performance. 

In [12]:
p = Portfolio(a, l)

In [13]:
p.get_var(ci=0.95, scenario="base", horizon=1)

[[[ 0.03382044  0.18408569  0.09266241 ...  0.03554729  0.088764
    0.07617755]]

 [[ 0.01679259 -0.04550061 -0.09428893 ...  0.029622    0.07422417
   -0.0022912 ]]

 [[ 0.01309374  0.02695028 -0.01554828 ... -0.06996182  0.03218013
    0.03762941]]

 ...

 [[ 0.02701042  0.07663768  0.11663636 ...  0.14656242  0.02642349
   -0.00548263]]

 [[ 0.02968172  0.0606776   0.09247812 ...  0.24559863  0.00930521
    0.21575426]]

 [[ 0.02204575  0.04976107 -0.05185413 ... -0.01370659 -0.02566822
   -0.10544129]]]


233601656.00722528

In [14]:
float(p.funded_ratio_vol(horizon=30, scenario="base"))

[[[ 1.52413543e-02  6.50447674e-02 -1.40164272e-02 ...  2.00252904e-01
    2.19926868e-01  4.17326915e-02]
  [ 2.87560856e-02 -2.22444493e-02  4.60318766e-01 ...  4.78497804e-01
    2.12458296e-01  1.04824290e+00]
  [-2.08788253e-03  4.52008212e-02  2.53968642e-01 ...  3.43084697e-02
    8.22940770e-02  2.00901024e-01]
  ...
  [ 1.73330436e-02 -1.08870869e-01 -2.60930455e-02 ...  1.76559422e-02
    1.27488933e-03  2.10410109e-02]
  [ 3.56110006e-02  7.64926518e-02  1.36718204e-01 ...  3.66590184e-02
    1.31627452e-01  1.16210241e-01]
  [ 2.36581590e-02  2.95456051e-02  4.78782734e-02 ...  1.66938957e-02
   -9.79316936e-02  6.21127129e-02]]

 [[ 1.43354119e-02  3.82692449e-02  4.76074083e-02 ...  9.43443327e-02
    1.08271012e-01  1.16642037e-01]
  [ 2.92191733e-02 -1.61406375e-02 -1.72487385e-01 ... -2.86630752e-02
    1.77180964e-02 -1.08473305e-01]
  [ 2.42301711e-02  3.32130231e-02  1.36121437e-01 ...  2.14477087e-01
    1.24433976e-01  9.87681593e-02]
  ...
  [ 2.39350885e-02  1.6

2.032311905649408

In [15]:
p.get_cvar(ci=0.95, scenario="base")

[[[ 0.03592454 -0.0441432  -0.0305824  ...  0.29449054  0.08770731
    0.20168693]]

 [[ 0.04830255  0.03974691  0.18189599 ... -0.1149678   0.11303052
    0.21189874]]

 [[ 0.02849529 -0.01449611  0.00533835 ...  0.12357547  0.0426827
   -0.10780414]]

 ...

 [[ 0.00079827 -0.03281062  0.25821788 ...  0.20447352  0.17617651
    0.28219302]]

 [[-0.00054346  0.01188344 -0.05723475 ...  0.22270938 -0.05749338
   -0.05693357]]

 [[ 0.0341327   0.0242148   0.18751055 ... -0.01118779  0.06612319
    0.03686304]]]


287276200.47176355

In [16]:
p.underfunding_probability()

[[[ 9.56673193e-03  4.41025412e-02  1.09124478e-01 ...  1.14318060e-01
    1.54534757e-01  3.70855552e-01]
  [-2.06844729e-02 -3.15529198e-02 -4.25645247e-02 ... -5.73661833e-02
   -5.37086331e-02  8.53306722e-02]
  [ 1.41853515e-02  6.80818588e-02  1.14542476e-01 ...  3.54278136e-01
    2.15174948e-01  9.38639139e-02]
  ...
  [ 4.23826767e-02  1.55068378e-01  1.96784487e-01 ... -8.97430149e-02
    1.08346266e-01 -1.59350824e-01]
  [ 5.11815224e-02  1.13110775e-01  7.11246865e-02 ... -1.61250437e-01
   -1.09297941e-02 -2.37878344e-01]
  [ 4.93003124e-03  1.77999006e-02  3.09342844e-01 ...  2.42334617e-01
    2.18986752e-01  5.26094662e-01]]

 [[ 3.16838205e-02  1.24272386e-01  4.30836991e-01 ...  1.09016065e-01
    2.96317381e-01  4.13259962e-01]
  [ 4.82190178e-02 -3.59440201e-02 -1.81501748e-01 ...  9.77541027e-02
   -1.27401601e-01  5.38143781e-02]
  [ 1.74885603e-03 -1.78350627e-02  1.16400890e-01 ... -1.27399931e-02
    2.43891145e-01  4.33965174e-01]
  ...
  [ 2.35272174e-02  9.2

0.6131

In [17]:
float(p.portfolio_avg())

4571098549.789394

In [18]:
float(p.funded_ratio_avg())

1.3122428486220745

## Scenarios
To model scenarios, the `Portfolio` object loads in with base scenarios:
* base
* bull
* bear
* stagflation
* gfc

But we also have the ability to create new scenarios

In [19]:
my_scenario = {
    "returns" : {
        "cash" : 0.0,
        "fixed_income" : 0.01,
        "can_equity" : -0.03,
        "us_equity" : -0.05,
        "id_equity" : -0.03,
        "em_equity" : -0.02,
        "real_estate" : 0.0,
        "infrastructure" : 0.0,
        "private_equity" : -0.06,
    } 
}
p.add_scenario("test_scenario_1", my_scenario)

In [20]:
p.get_var(scenario="test_scenario_1")

[[[ 0.00765262  0.0966341   0.19447022 ...  0.2436369   0.06108194
    0.189021  ]]

 [[ 0.04540924  0.01793958 -0.02142647 ...  0.10341206  0.10482403
    0.17242235]]

 [[ 0.02411204  0.04847777 -0.06470951 ...  0.11502367  0.08618072
   -0.10520341]]

 ...

 [[ 0.00796165  0.06072652 -0.05747466 ...  0.04733367  0.07107043
   -0.06551077]]

 [[ 0.00930531 -0.04996599  0.28220044 ...  0.1695385   0.14653304
    0.21007586]]

 [[ 0.02628075  0.03263775  0.1260679  ...  0.10617369  0.14544424
    0.10833154]]]


271971447.67589617

## Discussion Points

### Monte Carlo Methodology

The excel model uses a simple random return generation method that uses the portfolio's expected returns and variance to generate on return path for simulation. 

The current python approach uses brownian motion to project asset paths individually and then correlate them back to eachother.

These yield very different results:

Excel P(underfunding) ~ 0.15

Python  P(underfunding) ~ 0.8

The Brownian motion approach is industry standard but may not be necessary for our purposes. It assumes continuous price movements and constant returns + volatility. There are more sophisticated approaches to our portfolio setup that might be better. We may want to model different asset classes with different distributions

I have an early implementation of the Excel version, but want to consult before moving forward with that. This one is simpler and easier to understand + maintain
